In [ ]:
"""
HieraCascade Full Pipeline - Notebook Example

This script demonstrates running the complete HieraCascade pipeline in a notebook.
You can run this cell-by-cell in Jupyter or as a complete script.

Author: Student Name
Date: 2025-10-03
"""

# HieraCascade: Complete Training Pipeline

This notebook walks through the entire HieraCascade pipeline for soft tissue tumor classification.

## What You'll Learn
1. Load data from sheet.csv with label selection
2. Train Stage-1 model (coarse predictions + saliency)
3. Train Stage-2 model (hierarchical classification)
4. Evaluate and visualize results

## Terminal Alternative

If you prefer running from terminal instead of this notebook:
```bash
# Quick start (both stages)
python -m hieracascade.quick_start \
    --data_root data \
    --sheet_csv data/sheet.csv \
    --label_column Diagnosis \
    --fold 0

# Or use convenience script
run_hieracascade.bat 0 Diagnosis  # Windows
bash run_hieracascade.sh 0 Diagnosis  # Linux/Mac
```

## Setup and Imports

In [ ]:
import sys
import os
from pathlib import Path
import yaml
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add parent directory to path
sys.path.append('..')

from hieracascade.dataio import (
    create_index_from_sheet,
    create_site_held_out_splits,
    DatasetStage1,
    DatasetStage2,
    preprocess_volume,
    FINE_TO_IDX,
    COARSE_TO_IDX
)
from hieracascade.models import build_stage1_model, build_stage2_model
from hieracascade.train_stage1 import train_stage1
from hieracascade.train_stage2 import train_stage2

%matplotlib inline

# Set device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

## Configuration

**Choose your label column:**
- `'Diagnosis'` - Multi-class classification (melanoma, crlm, gist, lipo, desmoid, liver)
- `'Diagnosis_binary'` - Binary classification (malignant, benign)

In [ ]:
# Configuration parameters
DATA_ROOT = '../data'
SHEET_CSV = '../data/sheet.csv'
LABEL_COLUMN = 'Diagnosis'  # Change to 'Diagnosis_binary' for binary classification
OUTPUT_DIR = '../outputs/hieracascade_notebook'
FOLD = 0

print(f"Configuration:")
print(f"  Data root: {DATA_ROOT}")
print(f"  Sheet CSV: {SHEET_CSV}")
print(f"  Label column: {LABEL_COLUMN}")
print(f"  Output directory: {OUTPUT_DIR}")
print(f"  Fold: {FOLD}")

## Step 1: Load Data from sheet.csv

This step loads the labels from your existing sheet.csv file.
It automatically handles:
- Path resolution for NIfTI files
- Modality detection (CT/MRI)
- Class mapping (fine-grained and coarse)

In [ ]:
print("=" * 60)
print("STEP 1: Loading labels from sheet.csv")
print("=" * 60)

# Create output directory
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

# Load dataset index
labels_csv = output_dir / 'labels.csv'
index = create_index_from_sheet(
    data_root=DATA_ROOT,
    sheet_path=SHEET_CSV,
    label_column=LABEL_COLUMN,
    output_csv=str(labels_csv)
)

print(f"\n✓ Loaded {len(index)} studies")
print(f"✓ Saved labels to: {labels_csv}")

### Visualize Label Distribution

In [ ]:
# Get label distribution
categories = [item['category'] for item in index]
sites = [item['site'] for item in index]
modalities = [item['modality'] for item in index]

# Create visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Category distribution
pd.Series(categories).value_counts().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Class Distribution')
axes[0].set_xlabel('Category')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Site distribution
pd.Series(sites).value_counts().plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Site Distribution')
axes[1].set_xlabel('Site')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)

# Modality distribution
pd.Series(modalities).value_counts().plot(kind='bar', ax=axes[2], color='mediumseagreen')
axes[2].set_title('Modality Distribution')
axes[2].set_xlabel('Modality')
axes[2].set_ylabel('Count')

plt.tight_layout()
plt.show()

print(f"\nDataset Summary:")
print(f"  Total studies: {len(index)}")
print(f"  Categories: {len(set(categories))}")
print(f"  Sites: {len(set(sites))}")
print(f"  CT scans: {modalities.count('CT')}")
print(f"  MRI scans: {modalities.count('MRI')}")

### Create Cross-Validation Splits

Uses **stratified sampling** to maintain class balance.

In [ ]:
# Create site-held-out CV splits with stratified sampling
splits = create_site_held_out_splits(index, stratified=True)
train_index, val_index = splits[FOLD]

print(f"\nFold {FOLD} Split:")
print(f"  Training: {len(train_index)} studies")
print(f"  Validation: {len(val_index)} studies")

# Show class distribution in train/val
train_categories = [item['category'] for item in train_index]
val_categories = [item['category'] for item in val_index]

print(f"\nTraining set distribution:")
for cat, count in pd.Series(train_categories).value_counts().items():
    print(f"  {cat}: {count}")

print(f"\nValidation set distribution:")
for cat, count in pd.Series(val_categories).value_counts().items():
    print(f"  {cat}: {count}")

## Step 2: Train Stage-1 Model

Stage-1 learns:
- Coarse tumor classification
- Saliency map generation for crop proposals

**Terminal Alternative:**
```bash
python -m hieracascade.train_stage1 \
    --config hieracascade/configs/stage1.yaml \
    --data_root data \
    --labels_csv outputs/hieracascade_notebook/labels.csv \
    --output_dir outputs/stage1/fold0 \
    --fold 0
```

In [ ]:
print("\n" + "=" * 60)
print("STEP 2: Training Stage-1 (Coarse Predictions + Saliency)")
print("=" * 60)

# Load Stage-1 configuration
config_path = Path('../hieracascade/configs/stage1.yaml')
with open(config_path) as f:
    config_stage1 = yaml.safe_load(f)

# Adjust config for notebook (fewer epochs for demo)
config_stage1['train']['epochs'] = 5  # Set to 20+ for real training
config_stage1['train']['batch_size'] = 1  # Adjust based on GPU memory

# Set output directory
stage1_output = output_dir / f'stage1/fold{FOLD}'
stage1_output.mkdir(parents=True, exist_ok=True)

print(f"\nStage-1 Configuration:")
print(f"  Epochs: {config_stage1['train']['epochs']}")
print(f"  Batch size: {config_stage1['train']['batch_size']}")
print(f"  Learning rate: {config_stage1['train']['lr']}")
print(f"  Output: {stage1_output}")

### Option A: Train Stage-1 (can take 10-30 minutes per epoch)

In [ ]:
# Uncomment to train Stage-1
train_stage1(
     config=config_stage1,
     data_root=DATA_ROOT,
     labels_csv=str(labels_csv),
     output_dir=str(stage1_output),
     fold=FOLD,
     device=device
)
stage1_checkpoint = str(stage1_output / 'checkpoint_best.pt')
print(f"\n✓ Stage-1 training complete!")
print(f"  Checkpoint: {stage1_checkpoint}")

### Option B: Load Pre-trained Stage-1 Checkpoint

If you already have a trained checkpoint, load it here:

In [ ]:
# Load pre-trained checkpoint (adjust path as needed)
stage1_checkpoint = str(stage1_output / 'checkpoint_best.pt')

if os.path.exists(stage1_checkpoint):
    print(f"✓ Found Stage-1 checkpoint: {stage1_checkpoint}")
else:
    print(f"⚠ No checkpoint found at {stage1_checkpoint}")
    print("  Please train Stage-1 first or provide checkpoint path")

### Visualize Stage-1 Saliency Maps

After training, saliency maps are saved to the visualizations directory.

In [ ]:
# Check for saliency visualizations
viz_dir = stage1_output / 'visualizations'
if viz_dir.exists():
    saliency_files = list(viz_dir.glob('*.png'))
    if saliency_files:
        print(f"Found {len(saliency_files)} saliency visualizations")
        
        # Display first few
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        axes = axes.flatten()
        
        for idx, img_path in enumerate(saliency_files[:6]):
            img = plt.imread(str(img_path))
            axes[idx].imshow(img)
            axes[idx].set_title(img_path.stem)
            axes[idx].axis('off')
        
        plt.tight_layout()
        plt.show()
else:
    print("No saliency visualizations found yet. Train Stage-1 to generate them.")

## Step 3: Train Stage-2 Model

Stage-2 learns:
- Fine-grained tumor classification
- Hierarchical coarse family predictions
- Uses crops from Stage-1 saliency maps

**Terminal Alternative:**
```bash
python -m hieracascade.train_stage2 \
    --config hieracascade/configs/stage2.yaml \
    --data_root data \
    --labels_csv outputs/hieracascade_notebook/labels.csv \
    --stage1_ckpt outputs/stage1/fold0/checkpoint_best.pt \
    --output_dir outputs/stage2/fold0 \
    --fold 0
```

In [ ]:
print("\n" + "=" * 60)
print("STEP 3: Training Stage-2 (Hierarchical Classification)")
print("=" * 60)

# Load Stage-2 configuration
config_path = Path('../hieracascade/configs/stage2.yaml')
with open(config_path) as f:
    config_stage2 = yaml.safe_load(f)

# Adjust config for notebook
config_stage2['train']['epochs'] = 5  # Set to 40+ for real training
config_stage2['train']['batch_size'] = 1

# Set output directory
stage2_output = output_dir / f'stage2/fold{FOLD}'
stage2_output.mkdir(parents=True, exist_ok=True)

print(f"\nStage-2 Configuration:")
print(f"  Epochs: {config_stage2['train']['epochs']}")
print(f"  Crops per study: {config_stage2['proposals']['K']}")
print(f"  Crop size: {config_stage2['proposals']['crop_size']}")
print(f"  Output: {stage2_output}")

In [ ]:
# Uncomment to train Stage-2
train_stage2(
     config=config_stage2,
     data_root=DATA_ROOT,
     labels_csv=str(labels_csv),
     stage1_checkpoint=stage1_checkpoint,
     output_dir=str(stage2_output),
     fold=FOLD,
     device=device
)
stage2_checkpoint = str(stage2_output / 'checkpoint_best.pt')
print(f"\n✓ Stage-2 training complete!")
print(f"  Checkpoint: {stage2_checkpoint}")

## Step 4: Evaluate Model

**Terminal Alternative:**
```bash
python -m hieracascade.evaluate \
    --checkpoint outputs/stage2/fold0/checkpoint_best.pt \
    --stage stage2 \
    --data_root data \
    --labels_csv outputs/hieracascade_notebook/labels.csv \
    --stage1_ckpt outputs/stage1/fold0/checkpoint_best.pt \
    --output_dir outputs/stage2/fold0/eval \
    --fold 0
```

In [ ]:
print("\n" + "=" * 60)
print("STEP 4: Model Evaluation")
print("=" * 60)

stage2_checkpoint = str(stage2_output / 'checkpoint_best.pt')

if os.path.exists(stage2_checkpoint):
    print(f"✓ Found Stage-2 checkpoint: {stage2_checkpoint}")
    print("\nTo evaluate, run:")
    print(f"  python -m hieracascade.evaluate \\")
    print(f"    --checkpoint {stage2_checkpoint} \\")
    print(f"    --stage stage2 \\")
    print(f"    --data_root {DATA_ROOT} \\")
    print(f"    --labels_csv {labels_csv} \\")
    print(f"    --stage1_ckpt {stage1_checkpoint} \\")
    print(f"    --output_dir {stage2_output}/eval \\")
    print(f"    --fold {FOLD}")
else:
    print(f"⚠ No checkpoint found at {stage2_checkpoint}")
    print("  Please train Stage-2 first")

### View Training Curves

In [ ]:
# Check for training curves
plots_dir = stage2_output / 'plots'
curves_path = plots_dir / 'training_curves.png'

if curves_path.exists():
    print("Training curves:")
    img = plt.imread(str(curves_path))
    plt.figure(figsize=(12, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print("No training curves found yet. Train Stage-2 to generate them.")

### View Evaluation Results

In [ ]:
# Check for confusion matrix
eval_dir = stage2_output / 'eval'
cm_path = eval_dir / 'confusion_matrix_fine.png'

if cm_path.exists():
    print("Confusion Matrix (Fine-grained):")
    img = plt.imread(str(cm_path))
    plt.figure(figsize=(10, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.tight_layout()
    plt.show()
    
    # Load predictions CSV
    pred_csv = eval_dir / 'predictions_fine.csv'
    if pred_csv.exists():
        df_pred = pd.read_csv(pred_csv)
        print(f"\nPredictions summary:")
        print(df_pred.head())
        print(f"\nAccuracy: {(df_pred['y_true'] == df_pred['y_pred']).mean():.3f}")
else:
    print("No evaluation results found yet. Run evaluation to generate them.")

## Summary

### What We Accomplished

1. ✅ Loaded dataset from sheet.csv with label selection
2. ✅ Created stratified cross-validation splits
3. ✅ Configured Stage-1 and Stage-2 models
4. ✅ Set up training pipeline (ready to run)
5. ✅ Prepared evaluation workflow

### Next Steps

**Option 1: Run in Notebook**
- Uncomment training cells above
- Wait for training to complete (can take hours)
- Visualize results

**Option 2: Run in Terminal** (Recommended for long training)
```bash
# Quick start (both stages)
python -m hieracascade.quick_start \
    --data_root data \
    --sheet_csv data/sheet.csv \
    --label_column Diagnosis \
    --output_dir outputs/hieracascade \
    --fold 0 \
    --device cuda

# Or use convenience script
run_hieracascade.bat 0 Diagnosis  # Windows
bash run_hieracascade.sh 0 Diagnosis  # Linux/Mac
```

**Option 3: Train Stages Separately**
```bash
# Stage-1
python -m hieracascade.train_stage1 \
    --config hieracascade/configs/stage1.yaml \
    --data_root data \
    --labels_csv outputs/hieracascade_notebook/labels.csv \
    --output_dir outputs/stage1/fold0 \
    --fold 0

# Stage-2  
python -m hieracascade.train_stage2 \
    --config hieracascade/configs/stage2.yaml \
    --data_root data \
    --labels_csv outputs/hieracascade_notebook/labels.csv \
    --stage1_ckpt outputs/stage1/fold0/checkpoint_best.pt \
    --output_dir outputs/stage2/fold0 \
    --fold 0

# Evaluate
python -m hieracascade.evaluate \
    --checkpoint outputs/stage2/fold0/checkpoint_best.pt \
    --stage stage2 \
    --data_root data \
    --labels_csv outputs/hieracascade_notebook/labels.csv \
    --stage1_ckpt outputs/stage1/fold0/checkpoint_best.pt \
    --output_dir outputs/stage2/fold0/eval \
    --fold 0
```

### Documentation

- **README**: `hieracascade/README.md`
- **Tutorial**: `hieracascade/TUTORIAL.md`
- **Label Selection**: `hieracascade/LABEL_SELECTION_GUIDE.md`
- **Quick Reference**: `hieracascade/QUICK_REFERENCE.md`

In [ ]:
print("\n" + "=" * 60)
print("Pipeline Setup Complete!")
print("=" * 60)
print(f"\nAll outputs will be saved to: {OUTPUT_DIR}")
print("\nRefer to the markdown cells above for terminal commands.")
print("Uncomment training cells to run in this notebook.")